<a href="https://colab.research.google.com/github/elvissoares/EQE595-SimMol/blob/main/notebooks/8_Analise_Proteina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula Prática 08 - Análise de Simulações de Proteínas

Autor: [Prof. Elvis do A. Soares](https://github.com/elvissoares)

Contato: [elvis@peq.coppe.ufrj.br](mailto:elvis@peq.coppe.ufrj.br) - [Programa de Engenharia Química, PEQ/COPPE, UFRJ, Brasil](https://www.peq.coppe.ufrj.br/)

---

## 🎯 Objetivos da aula

Na Aula 07 preparamos e simulamos a lisozima em água (pH 7, 0,15 M de NaCl) e gravamos os resultados no Google Drive. Hoje vamos **analisar** esses resultados. Ao final da aula você deverá ser capaz de:

1. Calcular médias de grandezas termodinâmicas **com suas incertezas**;
2. Carregar e manipular trajetórias com o **MDTraj** e visualizá-las com o **py3Dmol**;
3. Caracterizar a estrutura da proteína ao longo do tempo: **RMSD**, **RMSF**, **raio de giração** e **ligações de hidrogênio**;
4. Organizar essas análises em funções reutilizáveis para o **projeto final**.

## 🗺️ Roteiro

| Parte | O que fazemos | Precisa da simulação? |
|---|---|---|
| 0 | Instalação e acesso aos arquivos da Aula 07 no Drive | — |
| 1 | Grandezas termodinâmicas e incertezas | `lisozima_log.csv` |
| 2 | Trajetória: carregando e visualizando | `lisozima.dcd` |
| 3 | RMSD: estabilidade global | trajetória |
| 4 | RMSF: flexibilidade por resíduo | trajetória |
| 5 | Raio de giração: compactação | trajetória |
| 6 | Ligações de hidrogênio | trajetória |
| 7 | Resumo e hands-on | — |

---
# Parte 0 — Preparando o ambiente

As análises rodam bem na CPU: não é preciso ativar a GPU no Colab. Também não precisamos do OpenMM: só do MDTraj (análise) e do py3Dmol (visualização).

In [ ]:
%%capture
!pip install mdtraj py3Dmol

In [ ]:
import os
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mdtraj as md
import py3Dmol

## 0.1 Os arquivos da Aula 07

Conectamos o Google Drive e verificamos se os arquivos gravados na Aula 07 estão lá. Os resultados desta aula (tabelas e figuras) vão para uma pasta nova, `EQE595/Aula08`.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    pasta_base = "/content/drive/MyDrive"
except ImportError:
    pasta_base = "."

pasta_dados = os.path.join(pasta_base, "EQE595", "Aula07")
pasta_saida = os.path.join(pasta_base, "EQE595", "Aula08")
os.makedirs(pasta_saida, exist_ok=True)

def dados(nome):
    return os.path.join(pasta_dados, nome)

def saida(nome):
    return os.path.join(pasta_saida, nome)

necessarios = ["1AKI.pdb", "lisozima_inicial.pdb", "lisozima.dcd", "lisozima_log.csv", "system_npt.xml"]
for nome in necessarios:
    if os.path.exists(dados(nome)):
        print(f"   ✅ {nome:22s} {os.path.getsize(dados(nome)) / 1e6:8.2f} MB")
    else:
        print(f"   ❌ {nome:22s} não encontrado: execute a Aula 07 primeiro")

Os parâmetros abaixo devem ser **os mesmos usados na Aula 07**. Eles dizem a que tempo corresponde cada quadro da trajetória.

In [ ]:
temperatura    = 298.15     # K
pressao        = 1.0        # bar
t_nvt          = 40.0       # ps de equilibração NVT
dt_quadro      = 2.0        # ps entre quadros da trajetória (intervalo_traj × dt = 500 × 4 fs)

---
# Parte 1 — Grandezas termodinâmicas e suas incertezas

## 1.1 Séries temporais

Lemos o arquivo CSV gravado pelo `StateDataReporter`. Se a simulação da Aula 07 foi retomada de um *checkpoint*, alguns passos aparecem duas vezes; removemos as repetições pelo número do passo.

Também calculamos a **entalpia** instantânea, $H = E + PV$, onde $E$ é a energia total (cinética + potencial).

In [ ]:
df = pd.read_csv(dados("lisozima_log.csv"))
df = df.drop_duplicates(df.columns[0], keep="last").reset_index(drop=True)

NA = 6.02214076e23
PV = pressao * 1e5 * 1e-27 * NA / 1e3        # P × (1 nm³) em kJ/mol:  bar → Pa, nm³ → m³, J → kJ
df["Enthalpy (kJ/mole)"] = df["Total Energy (kJ/mole)"] + PV * df["Box Volume (nm^3)"]

print(f"{len(df)} registros, de {df['Time (ps)'].min():.0f} a {df['Time (ps)'].max():.0f} ps")
df.head()

In [ ]:
grandezas = {
    "Potential Energy (kJ/mole)": (r"$U$ (kJ/mol)", "C0"),
    "Temperature (K)":            (r"$T$ (K)", "C3"),
    "Box Volume (nm^3)":          (r"$V$ (nm³)", "C1"),
    "Density (g/mL)":             (r"$\rho$ (g/mL)", "C2"),
}
fig, axs = plt.subplots(len(grandezas), 1, figsize=(8, 7), sharex=True)
for ax, (coluna, (rotulo, cor)) in zip(axs, grandezas.items()):
    ax.plot(df["Time (ps)"], df[coluna], color=cor, lw=0.6)
    ax.axvline(t_nvt, color="k", ls="--", lw=1)
    ax.set_ylabel(rotulo)
axs[-1].set_xlabel("Tempo (ps)")
plt.tight_layout()

> ❓ A partir de que tempo as grandezas flutuam em torno de um valor constante? Os dados anteriores são a **equilibração** e não devem entrar nas médias. Escolha `t_corte` olhando os gráficos. Na Parte 3 vamos conferir essa escolha com o RMSD.

In [ ]:
t_corte = 100.0    # ps
prod = df[df["Time (ps)"] > t_corte]
dt_registro = df["Time (ps)"].diff().median()      # ps entre registros do CSV
print(f"Produção: {len(prod)} registros ({prod['Time (ps)'].iloc[-1] - prod['Time (ps)'].iloc[0]:.0f} ps), um a cada {dt_registro:g} ps")

## 1.2 Média, flutuação e erro da média

Como vimos na Aula 06, a dispersão $\sigma$ dos valores instantâneos **não** é a incerteza da média. Amostras sucessivas são **correlacionadas**, por isso usamos a **média em blocos**: dividimos a série em blocos de duração $\tau_b$ e estimamos o erro a partir da dispersão das médias dos blocos. Quando os blocos ficam mais longos que o tempo de correlação, a estimativa atinge um **platô**.

In [ ]:
def erro_por_blocos(x, tamanho_bloco):
    # Erro da média de x usando blocos de 'tamanho_bloco' amostras
    x = np.asarray(x)
    n_blocos = len(x) // tamanho_bloco
    medias = x[:n_blocos * tamanho_bloco].reshape(n_blocos, tamanho_bloco).mean(axis=1)
    return medias.std(ddof=1) / np.sqrt(n_blocos)

def curva_de_blocos(x, dt, n_min_blocos=8):
    # Erro da média em função da duração do bloco (mantendo pelo menos n_min_blocos blocos)
    tamanhos = np.unique(np.logspace(0, np.log10(len(x) // n_min_blocos), 25).astype(int))
    return tamanhos * dt, np.array([erro_por_blocos(x, b) for b in tamanhos])

fig, axs = plt.subplots(1, len(grandezas), figsize=(14, 3))
for ax, (coluna, (rotulo, cor)) in zip(axs, grandezas.items()):
    tau, erro = curva_de_blocos(prod[coluna], dt_registro)
    ax.semilogx(tau, erro, "o-", color=cor, ms=3)
    ax.set_xlabel("Duração do bloco (ps)"); ax.set_title(rotulo, fontsize=10)
axs[0].set_ylabel("Erro da média")
plt.tight_layout()

Escolha uma duração de bloco no **platô** dos gráficos acima e monte a tabela de resultados:

In [ ]:
bloco_ps = 50.0       # ps
b = max(1, int(bloco_ps / dt_registro))

linhas = []
for coluna in list(grandezas) + ["Enthalpy (kJ/mole)"]:
    x = prod[coluna].values
    erro = erro_por_blocos(x, b)
    linhas.append({"grandeza": coluna, "média": x.mean(), "erro da média": erro,
                   "flutuação σ": x.std(), "amostras independentes ≈": (x.std() / erro)**2})
tabela_termo = pd.DataFrame(linhas).set_index("grandeza")
tabela_termo.to_csv(saida("termodinamica.csv"))
tabela_termo.style.format("{:.4g}")

> ❓ Escreva cada resultado na forma $\langle x\rangle \pm \delta x$ com o número correto de algarismos significativos. Quantas amostras **independentes** você realmente tem em 1 ns? Para reduzir o erro da média à metade, quanto tempo de simulação seria necessário?

## 1.3 Física estatística a partir das flutuações

As flutuações também trazem informação física (Aula 06). O `System` gravado na Aula 07 (`system_npt.xml`) é um arquivo de texto em XML. Contando nele as partículas e os vínculos, obtemos o número de graus de liberdade:

$$ \frac{\sigma_T}{\langle T\rangle} = \sqrt{\frac{2}{N_\text{gl}}}, \qquad N_\text{gl} = 3N_\text{átomos} - N_\text{vínculos} - 3, \qquad
\kappa_T = \frac{\langle V^2\rangle - \langle V\rangle^2}{k_B T\,\langle V\rangle}. $$

In [ ]:
# O System gravado na Aula 07 é um arquivo XML: contamos as partículas e os vínculos
raiz = ET.parse(dados("system_npt.xml")).getroot()
n_atomos = len(raiz.find("Particles"))
n_vinculos = len(raiz.find("Constraints"))
n_gl = 3 * n_atomos - n_vinculos - 3

T = prod["Temperature (K)"]
print(f"Átomos: {n_atomos},  vínculos: {n_vinculos},  N_gl = {n_gl}")
print(f"σ_T/<T>: simulado = {T.std() / T.mean():.5f},   teórico √(2/N_gl) = {np.sqrt(2 / n_gl):.5f}")

kB = 1.380649e-23
V = prod["Box Volume (nm^3)"].values * 1e-27
kappa = V.var() / (kB * temperatura * V.mean()) * 1e10
print(f"κ_T = {kappa:.1f} × 10⁻⁵ bar⁻¹   (água pura experimental ≈ 4,5 × 10⁻⁵ bar⁻¹)")

> ❓ O $\sigma_T$ concorda com a previsão? E o $\kappa_T$? Lembre que variâncias convergem bem mais devagar que médias: com 1 ns, a incerteza de $\kappa_T$ é grande.

---
# Parte 2 — A trajetória

## 2.1 Carregando com o MDTraj

O arquivo DCD guarda **só as coordenadas** e a caixa de cada quadro. Os nomes dos átomos, os resíduos e as ligações vêm da **topologia**, lida do PDB inicial. Um objeto `md.Trajectory` reúne as duas coisas:

| Atributo | Conteúdo |
|---|---|
| `traj.xyz` | coordenadas, array (quadros × átomos × 3), em **nm** |
| `traj.topology` | átomos, resíduos, cadeias e ligações |
| `traj.unitcell_lengths` | arestas da caixa em cada quadro |
| `traj.n_frames`, `traj.n_atoms` | tamanhos |

In [ ]:
traj = md.load(dados("lisozima.dcd"), top=dados("lisozima_inicial.pdb"))
t_traj = (np.arange(traj.n_frames) + 1) * dt_quadro          # ps
print(traj)
print(f"Memória ocupada pelas coordenadas: {traj.xyz.nbytes / 1e6:.0f} MB")

O MDTraj tem uma **linguagem de seleção** para escolher átomos, parecida com a de programas como o VMD:

| Seleção | Átomos |
|---|---|
| `"protein"` | todos os átomos da proteína |
| `"backbone"` | cadeia principal: N, Cα, C, O |
| `"name CA"` | carbonos α |
| `"water and name O"` | oxigênios da água |
| `"resname NA or resname CL"` | íons |
| `"protein and not element H"` | átomos pesados da proteína |

In [ ]:
top = traj.topology
for selecao in ["protein", "backbone", "name CA", "water and name O", "resname NA or resname CL"]:
    print(f"   {selecao:28s} → {len(top.select(selecao)):6d} átomos")

idx_proteina = top.select("protein")
traj_prot = traj.atom_slice(idx_proteina)        # só a proteína: análises mais leves
ref = md.load(dados("lisozima_inicial.pdb"), atom_indices=idx_proteina)   # estrutura do cristal (com H)

## 2.2 Visualizando a trajetória

Com o py3Dmol podemos animar a trajetória. Para manter a animação leve, mostramos só os átomos pesados da proteína, num quadro a cada 10, depois de **sobrepor** todos os quadros ao primeiro (removendo a translação e a rotação da molécula inteira).

In [ ]:
pesados = traj_prot.topology.select("not element H")
filme = traj_prot[::10].atom_slice(pesados)
filme.superpose(filme, 0)
filme.save_pdb("filme.pdb")

vis = py3Dmol.view(width=600, height=450)
vis.addModelsAsFrames(open("filme.pdb").read(), "pdb")
vis.setStyle({"cartoon": {"color": "spectrum"}})
vis.zoomTo()
vis.animate({"loop": "forward", "interval": 150})
vis.show()

---
# Parte 3 — RMSD: a proteína manteve a estrutura?

O **RMSD** (*root-mean-square deviation*) mede o quanto uma estrutura difere de uma referência, **depois** de sobrepor as duas da melhor forma possível (translação + rotação ótimas):

$$\text{RMSD}(t) = \min_{\text{rot., transl.}} \sqrt{\frac{1}{N}\sum_{i=1}^N \left|\mathbf r_i(t) - \mathbf r_i^\text{ref}\right|^2}.$$

Usamos como referência a **estrutura cristalográfica** e como átomos a cadeia principal (*backbone*), que define o enovelamento. Um RMSD que cresce e depois se estabiliza num **platô** indica que a proteína relaxou a partir do cristal e passou a flutuar em torno de uma estrutura estável.

In [ ]:
backbone = traj_prot.topology.select("backbone")
rmsd = md.rmsd(traj_prot, ref, atom_indices=backbone) * 10          # Å
rmsd_CA_inicio = md.rmsd(traj_prot, traj_prot, 0, atom_indices=traj_prot.topology.select("name CA")) * 10

plt.figure(figsize=(8, 3.5))
plt.plot(t_traj, rmsd, label="backbone × cristal")
plt.plot(t_traj, rmsd_CA_inicio, alpha=0.7, label="Cα × 1º quadro da simulação")
plt.axvline(t_nvt, color="k", ls="--", lw=1); plt.axvline(t_corte, color="gray", ls=":", lw=1)
plt.xlabel("Tempo (ps)"); plt.ylabel("RMSD (Å)"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(saida("rmsd.png"), dpi=150)

producao = t_traj > t_corte
b_traj = max(1, int(bloco_ps / dt_quadro))
print(f"RMSD médio na produção: {rmsd[producao].mean():.2f} ± {erro_por_blocos(rmsd[producao], b_traj):.2f} Å")

> ❓ **Discuta:**
> - O RMSD atinge um platô? O `t_corte` escolhido na Parte 1 é compatível com ele?
> - Por que o RMSD não vai a zero, mesmo com a proteína estável? Como comparação: duas estruturas cristalográficas da mesma proteína, de laboratórios diferentes, costumam diferir em alguns décimos de Å.
> - Um RMSD de ~1–2 Å no backbone é típico de uma proteína globular estável. O que você esperaria ver se a proteína desnaturasse?

## 3.1 Visualizando: cristal × final da simulação

Sobrepomos o último quadro ao cristal e mostramos as duas estruturas juntas: cristal em cinza, simulação colorida.

In [ ]:
final = traj_prot[-1].superpose(ref, atom_indices=backbone)
ref.save_pdb("cristal.pdb"); final.save_pdb("final.pdb")

vis = py3Dmol.view(width=600, height=450)
vis.addModel(open("cristal.pdb").read(), "pdb")
vis.addModel(open("final.pdb").read(), "pdb")
vis.setStyle({"model": 0}, {"cartoon": {"color": "lightgray", "opacity": 0.8}})
vis.setStyle({"model": 1}, {"cartoon": {"color": "spectrum"}})
vis.zoomTo()
vis.show()

## 3.2 Mapa de RMSD entre quadros

Calculando o RMSD de **cada** quadro em relação a **cada** outro, obtemos uma matriz. Blocos escuros na diagonal indicam períodos em que a proteína ficou numa mesma conformação; transições entre conformações aparecem como mudanças de bloco.

In [ ]:
sub = traj_prot[producao][::2]
ca_sub = sub.topology.select("name CA")
mapa = np.array([md.rmsd(sub, sub, i, atom_indices=ca_sub) for i in range(sub.n_frames)]) * 10
t_sub = t_traj[producao][::2]

plt.figure(figsize=(5.5, 4.5))
plt.imshow(mapa, origin="lower", cmap="viridis", extent=[t_sub[0], t_sub[-1], t_sub[0], t_sub[-1]])
plt.colorbar(label="RMSD dos Cα (Å)")
plt.xlabel("Tempo (ps)"); plt.ylabel("Tempo (ps)"); plt.tight_layout()

---
# Parte 4 — RMSF: quais regiões são flexíveis?

O RMSD diz **quanto** a proteína como um todo se afasta da referência. O **RMSF** (*root-mean-square fluctuation*) diz **onde** estão os movimentos: para cada átomo, mede a amplitude da flutuação em torno da sua posição média, depois de sobrepor todos os quadros:

$$\text{RMSF}_i = \sqrt{\left\langle \left|\mathbf r_i(t) - \langle\mathbf r_i\rangle\right|^2\right\rangle_t}.$$

Usamos os carbonos α (um por resíduo) e apenas a produção.

O fator B do cristal está relacionado ao deslocamento quadrático médio por $B = \frac{8\pi^2}{3}\langle \Delta r^2 \rangle$. Assim podemos comparar a simulação **diretamente** com o experimento:

$$\text{RMSF}_B = \sqrt{\frac{3B}{8\pi^2}}.$$

In [ ]:
def le_bfactors_CA(arquivo):
    # Número do resíduo e fator B (Å²) dos átomos CA de um arquivo PDB
    residuos, B = [], []
    for linha in open(arquivo):
        if linha.startswith("ATOM") and linha[12:16].strip() == "CA":
            residuos.append(int(linha[22:26]))
            B.append(float(linha[60:66]))
    return np.array(residuos), np.array(B)

prod_prot = traj_prot[producao]
ca = prod_prot.topology.select("name CA")
prod_prot.superpose(prod_prot, 0, atom_indices=ca)

xyz = prod_prot.xyz[:, ca, :]
rmsf = np.sqrt(((xyz - xyz.mean(axis=0))**2).sum(axis=2).mean(axis=0)) * 10      # Å
residuos = np.array([prod_prot.topology.atom(i).residue.resSeq for i in ca])

res_cristal, B = le_bfactors_CA(dados("1AKI.pdb"))
rmsf_B = np.sqrt(3 * B / (8 * np.pi**2))

# Estrutura secundária da estrutura inicial (DSSP), para sombrear o gráfico
ss = md.compute_dssp(ref, simplified=True)[0]

fig, ax = plt.subplots(figsize=(10, 3.8))
for i, s in enumerate(ss):
    if s in "HE":
        ax.axvspan(residuos[i] - 0.5, residuos[i] + 0.5, color="C3" if s == "H" else "C0", alpha=0.12, lw=0)
ax.plot(residuos, rmsf, "-o", ms=3, label="Simulação (RMSF dos Cα)")
if B.max() > 0:
    ax.plot(res_cristal, rmsf_B, "-s", ms=3, label=r"Cristal: $\sqrt{3B/8\pi^2}$")
ax.set_xlabel("Resíduo"); ax.set_ylabel("Flutuação (Å)")
ax.set_title("fundo vermelho: hélice   fundo azul: folha β", fontsize=9)
ax.legend(fontsize=8, loc="upper left", framealpha=0.9); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(saida("rmsf.png"), dpi=150)

pd.DataFrame({"resíduo": residuos, "RMSF (Å)": rmsf}).to_csv(saida("rmsf.csv"), index=False)
print(f"RMSF médio: {rmsf.mean():.2f} Å")
if B.max() > 0 and len(rmsf) == len(rmsf_B):
    print(f"Correlação entre RMSF simulado e cristalográfico: {np.corrcoef(rmsf, rmsf_B)[0, 1]:.2f}")
print("Resíduos mais flexíveis:", [str(prod_prot.topology.atom(ca[i]).residue) for i in np.argsort(rmsf)[-5:][::-1]])

> ❓ **Discuta:**
> - Quais regiões são mais flexíveis: os terminais, as alças (*loops*) ou as hélices e folhas?
> - As curvas da simulação e do cristal têm o mesmo **padrão**? E a mesma **magnitude**? Lembre que o fator B também inclui a desordem estática do cristal (moléculas ligeiramente diferentes em cada cela unitária), e que a simulação de 1 ns só captura movimentos rápidos.

## 4.1 RMSF sobre a estrutura

Gravamos o RMSF de cada resíduo na coluna de fator B de um arquivo PDB e pedimos ao py3Dmol que pinte a estrutura por essa coluna: azul = rígido, vermelho = flexível.

In [ ]:
rmsf_por_atomo = np.array([rmsf[np.searchsorted(residuos, a.residue.resSeq)] for a in ref.topology.atoms])
ref.save_pdb("rmsf_estrutura.pdb", bfactors=rmsf_por_atomo)

vis = py3Dmol.view(width=600, height=450)
vis.addModel(open("rmsf_estrutura.pdb").read(), "pdb")
vis.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "rwb",
                                          "min": float(rmsf.max()), "max": float(rmsf.min())}}})
vis.zoomTo()
vis.show()

---
# Parte 5 — Raio de giração: a proteína ficou mais compacta?

O raio de giração mede o tamanho da proteína, ponderando cada átomo pela sua massa:

$$R_g = \sqrt{\frac{\sum_i m_i \left|\mathbf r_i - \mathbf r_\text{cm}\right|^2}{\sum_i m_i}}.$$

É a grandeza medida experimentalmente por espalhamento de raios X a baixo ângulo (SAXS). Uma proteína que desnatura **se expande**, e seu $R_g$ aumenta.

In [ ]:
massas = np.array([a.element.mass for a in traj_prot.topology.atoms])
rg = md.compute_rg(traj_prot, masses=massas) * 10            # Å
rg_cristal = md.compute_rg(ref, masses=massas)[0] * 10

plt.figure(figsize=(8, 3.5))
plt.plot(t_traj, rg, color="C2")
plt.axhline(rg_cristal, color="gray", ls="--", label=f"cristal ({rg_cristal:.2f} Å)")
plt.axvline(t_nvt, color="k", ls="--", lw=1); plt.axvline(t_corte, color="gray", ls=":", lw=1)
plt.xlabel("Tempo (ps)"); plt.ylabel(r"$R_g$ (Å)"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(saida("rg.png"), dpi=150)

print(f"R_g na produção: {rg[producao].mean():.2f} ± {erro_por_blocos(rg[producao], b_traj):.2f} Å "
      f"(flutuação σ = {rg[producao].std():.2f} Å)")

Uma estimativa simples: se a proteína fosse uma **esfera compacta** de massa molar $M$ e densidade $\rho_p \approx 1{,}35$ g/cm³ (típica de proteínas), seu raio seria $R = \left(\frac{3M}{4\pi\rho_p N_A}\right)^{1/3}$, e o raio de giração de uma esfera maciça é $R_g = \sqrt{3/5}\,R$.

In [ ]:
M = massas.sum()                                                   # g/mol
R = (3 * M / (4 * np.pi * 1.35 * 6.02214076e23)) ** (1 / 3) * 1e8  # Å
print(f"Massa molar: {M/1000:.1f} kDa")
print(f"Esfera compacta: R = {R:.1f} Å,  R_g = {np.sqrt(3/5) * R:.1f} Å")
print(f"Simulação:                  R_g = {rg[producao].mean():.1f} Å")

> ❓ O $R_g$ da simulação é maior ou menor que o da esfera compacta? O que isso diz sobre a forma da lisozima? O $R_g$ variou durante a simulação, ou a proteína manteve a compactação do cristal?

---
# Parte 6 — Ligações de hidrogênio

As ligações de hidrogênio sustentam a estrutura secundária (hélices e folhas β são mantidas por ligações de H entre grupos N–H e C=O da cadeia principal) e fazem a interface da proteína com a água.

Usamos um critério **geométrico**: há uma ligação de H entre um doador D–H e um aceitador A se

$$ d(\text{D}\cdots\text{A}) < 3{,}5\ \text{Å} \qquad \text{e} \qquad \angle(\text{H–D}\cdots\text{A}) < 30°. $$

Na proteína, os doadores são os N e O ligados a H. Como aceitadores, usamos os átomos de O. Pares dentro do mesmo resíduo são ignorados.

> 💡 O MDTraj também tem funções prontas (`md.baker_hubbard`, `md.wernet_nilsson`), com critérios um pouco diferentes. Escrevemos a nossa para deixar o critério explícito e para tratar a água de forma eficiente.

In [ ]:
def pares_doador_H(topologia, selecao):
    # Lista de pares (D, H) com D (N ou O) dentro da seleção
    sel = set(topologia.select(selecao))
    pares = []
    for a, b in topologia.bonds:
        for d, h in ((a, b), (b, a)):
            if d.index in sel and d.element.symbol in ("N", "O") and h.element.symbol == "H":
                pares.append((d.index, h.index))
    return np.array(pares).reshape(-1, 2)

def ligacoes_H(quadro, doadores, aceitadores, d_max=0.35, ang_max=30.0, residuo_de=None):
    # Ligações de H num quadro: retorna array (n, 3) com (D, H, A)
    D_unicos = np.unique(doadores[:, 0])
    if len(D_unicos) == 0 or len(aceitadores) == 0:
        return np.empty((0, 3), dtype=int)
    d_idx, a_idx = np.meshgrid(D_unicos, aceitadores, indexing="ij")
    pares = np.column_stack([d_idx.ravel(), a_idx.ravel()])
    pares = pares[pares[:, 0] != pares[:, 1]]
    if residuo_de is not None:                                    # ignora pares no mesmo resíduo
        pares = pares[residuo_de[pares[:, 0]] != residuo_de[pares[:, 1]]]
    perto = pares[md.compute_distances(quadro, pares)[0] < d_max]
    if len(perto) == 0:
        return np.empty((0, 3), dtype=int)
    H_de = {}
    for d, h in doadores:
        H_de.setdefault(d, []).append(h)
    triplas = np.array([(h, d, a) for d, a in perto for h in H_de[d]])     # ângulo medido em D
    ang = np.degrees(md.compute_angles(quadro, triplas)[0])
    return triplas[ang < ang_max][:, [1, 0, 2]]

top_prot = traj_prot.topology
doadores_prot = pares_doador_H(top_prot, "protein")
aceitadores_prot = top_prot.select("protein and element O")
residuo_de = np.array([a.residue.index for a in top_prot.atoms])
print(f"Doadores (D–H): {len(doadores_prot)},  aceitadores: {len(aceitadores_prot)}")

## 6.1 Ligações de H internas da proteína

In [ ]:
lig_por_quadro = [ligacoes_H(traj_prot[k], doadores_prot, aceitadores_prot, residuo_de=residuo_de)
                  for k in range(traj_prot.n_frames)]
n_lig = np.array([len(l) for l in lig_por_quadro])

def eh_cadeia_principal(d, a):
    return top_prot.atom(d).name == "N" and top_prot.atom(a).name == "O"
n_lig_cp = np.array([sum(eh_cadeia_principal(d, a) for d, h, a in l) for l in lig_por_quadro])

plt.figure(figsize=(8, 3.5))
plt.plot(t_traj, n_lig, label="total")
plt.plot(t_traj, n_lig_cp, label="cadeia principal (N–H···O=C)")
plt.axvline(t_nvt, color="k", ls="--", lw=1); plt.axvline(t_corte, color="gray", ls=":", lw=1)
plt.xlabel("Tempo (ps)"); plt.ylabel("Nº de ligações de H"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(saida("hbonds_internas.png"), dpi=150)

print(f"Ligações de H internas na produção: {n_lig[producao].mean():.1f} ± {erro_por_blocos(n_lig[producao], b_traj):.1f}")
print(f"   das quais na cadeia principal:   {n_lig_cp[producao].mean():.1f}")

Uma mesma ligação de H pode se romper e se refazer. A **ocupação** é a fração dos quadros da produção em que ela existe:

In [ ]:
contagem = Counter()
quadros_prod = np.where(producao)[0]
for k in quadros_prod:
    contagem.update(map(tuple, lig_por_quadro[k]))

ocupacao = pd.DataFrame([
    {"doador": f"{top_prot.atom(d).residue}-{top_prot.atom(d).name}",
     "aceitador": f"{top_prot.atom(a).residue}-{top_prot.atom(a).name}",
     "tipo": "cadeia principal" if eh_cadeia_principal(d, a) else "cadeia lateral",
     "ocupação": n_q / len(quadros_prod), "D": d, "H": h, "A": a}
    for (d, h, a), n_q in contagem.items()]).sort_values("ocupação", ascending=False)

print(f"Ligações de H distintas observadas: {len(ocupacao)}")
print(f"Com ocupação > 50 %: {(ocupacao['ocupação'] > 0.5).sum()}")
ocupacao.to_csv(saida("hbonds_ocupacao.csv"), index=False)
ocupacao.head(15)[["doador", "aceitador", "tipo", "ocupação"]]

Visualizando as ligações de H **persistentes** (ocupação > 50 %) no último quadro, como cilindros tracejados entre o H e o aceitador:

In [ ]:
quadro = traj_prot[-1]
quadro.save_pdb("quadro_final.pdb")
pos = quadro.xyz[0] * 10                                           # Å

vis = py3Dmol.view(width=600, height=450)
vis.addModel(open("quadro_final.pdb").read(), "pdb")
vis.setStyle({"cartoon": {"color": "lightgray", "opacity": 0.85}})
for _, l in ocupacao[ocupacao["ocupação"] > 0.5].iterrows():
    h, a = pos[int(l["H"])], pos[int(l["A"])]
    cor = "red" if l["tipo"] == "cadeia principal" else "blue"
    vis.addCylinder({"start": dict(x=float(h[0]), y=float(h[1]), z=float(h[2])),
                     "end": dict(x=float(a[0]), y=float(a[1]), z=float(a[2])),
                     "radius": 0.12, "color": cor, "dashed": True})
vis.zoomTo()
vis.show()
print("vermelho: cadeia principal (N–H···O=C);  azul: envolvendo cadeias laterais")

> ❓ **Discuta:**
> - Onde ficam as ligações de H da cadeia principal? Compare com as hélices e folhas β.
> - As ligações de H internas foram preservadas ao longo da simulação? Isso é consistente com o RMSD?

## 6.2 Ligações de H entre a proteína e a água

Na superfície, a proteína faz ligações de H com a água. Como há milhares de moléculas de água, primeiro selecionamos, em cada quadro, apenas as águas próximas da proteína (`md.compute_neighbors`) e depois aplicamos o critério geométrico nos dois sentidos: proteína doadora → água aceitadora e água doadora → proteína aceitadora. Para economizar tempo, usamos 1 a cada 5 quadros.

In [ ]:
O_agua = top.select("water and name O")
H_da_agua = {o: [a.index for a in top.atom(o).residue.atoms if a.element.symbol == "H"] for o in O_agua}
doadores_p = pares_doador_H(top, "protein")
aceitadores_p = top.select("protein and element O")
polares_p = top.select("protein and (element N or element O)")

def ligacoes_proteina_agua(q):
    O_perto = md.compute_neighbors(q, 0.35, polares_p, haystack_indices=O_agua)[0]
    doadores_w = np.array([(o, h) for o in O_perto for h in H_da_agua[o]]).reshape(-1, 2)
    return ligacoes_H(q, doadores_p, O_perto), ligacoes_H(q, doadores_w, aceitadores_p)

passo = 5
t_pa, n_pa_doa, n_pa_ace = [], [], []
for k in range(0, traj.n_frames, passo):
    prot_doa, agua_doa = ligacoes_proteina_agua(traj[k])
    t_pa.append(t_traj[k]); n_pa_doa.append(len(prot_doa)); n_pa_ace.append(len(agua_doa))
t_pa, n_pa_doa, n_pa_ace = map(np.array, (t_pa, n_pa_doa, n_pa_ace))
n_pa = n_pa_doa + n_pa_ace

plt.figure(figsize=(8, 3.5))
plt.plot(t_pa, n_pa, label="total")
plt.plot(t_pa, n_pa_doa, label="proteína doadora")
plt.plot(t_pa, n_pa_ace, label="água doadora")
plt.axvline(t_nvt, color="k", ls="--", lw=1); plt.axvline(t_corte, color="gray", ls=":", lw=1)
plt.xlabel("Tempo (ps)"); plt.ylabel("Nº de ligações de H proteína–água"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(saida("hbonds_agua.png"), dpi=150)

prod_pa = t_pa > t_corte
print(f"Ligações de H proteína–água na produção: {n_pa[prod_pa].mean():.0f} ± "
      f"{erro_por_blocos(n_pa[prod_pa], max(1, b_traj // passo)):.0f}")

> ❓ **Discuta:**
> - A proteína faz mais ligações de H com ela mesma ou com a água? Por quê?
> - Quem doa mais: a proteína ou a água? Compare o número de doadores e de aceitadores que cada uma oferece.

Visualizando a **camada de hidratação**: as águas que fazem ligação de H com a proteína no último quadro.

In [ ]:
q = traj[-1]
q.image_molecules(inplace=True, anchor_molecules=[set(top.atom(i) for i in idx_proteina)])
prot_doa, agua_doa = ligacoes_proteina_agua(q)
O_ligadas = set(prot_doa[:, 2]) | set(agua_doa[:, 0])
atomos = sorted(set(idx_proteina) | {a.index for o in O_ligadas for a in top.atom(o).residue.atoms})
q.atom_slice(atomos).save_pdb("hidratacao.pdb")

vis = py3Dmol.view(width=600, height=450)
vis.addModel(open("hidratacao.pdb").read(), "pdb")
vis.setStyle({"cartoon": {"color": "spectrum"}})
vis.addStyle({"resn": "HOH"}, {"sphere": {"radius": 0.5, "color": "red"}})
vis.zoomTo()
vis.show()
print(f"{len(O_ligadas)} moléculas de água ligadas à proteína neste quadro")

---
# Parte 7 — Resumo e hands-on

A tabela abaixo reúne os principais resultados. É a mesma tabela pedida no **projeto final**.

In [ ]:
resumo = pd.DataFrame([{
    "RMSD médio (Å)": rmsd[producao].mean(),
    "RMSF médio (Å)": rmsf.mean(),
    "Rg médio (Å)": rg[producao].mean(),
    "Nº médio de H-bonds internas": n_lig[producao].mean(),
    "Nº médio de H-bonds proteína–água": n_pa[prod_pa].mean(),
    "densidade (g/mL)": prod["Density (g/mL)"].mean(),
}], index=["lisozima, pH 7, 0,15 M"])
resumo.to_csv(saida("resumo.csv"))
resumo.round(3)

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON 1: o tempo de corte &#x1F528;
</span>**

> Refaça as médias da Parte 1 e o RMSD médio com `t_corte` = 50, 100, 200 e 400 ps. Os resultados mudam além das barras de erro? O que isso diz sobre a equilibração?

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON 2: uma função para o projeto final &#x1F528;
</span>**

> No projeto final cada grupo vai analisar várias simulações (forças iônicas e temperaturas diferentes). Junte as análises desta aula numa função `analisa_simulacao(pasta, t_corte)` que leia os arquivos de uma pasta (no formato da Aula 07) e retorne uma linha da tabela-resumo. Teste-a com a pasta da Aula 07.

---
## 📌 Resumo

- Toda média de simulação precisa de **incerteza**: descarte a equilibração e use a média em blocos.
- O MDTraj une **coordenadas** (DCD) e **topologia** (PDB) e oferece seleções de átomos, sobreposição, RMSD, $R_g$ e muito mais. O py3Dmol permite ver e animar os resultados.
- **RMSD**: estabilidade global. **RMSF**: flexibilidade local (comparável ao fator B). **$R_g$**: compactação. **Ligações de H**: estrutura secundária e interação com a água.
- Uma simulação de 1 ns mostra flutuações em torno da estrutura nativa. Processos como desnaturação e enovelamento exigem escalas de tempo muito maiores.

### Para saber mais
- [MDTraj: documentação e exemplos](https://www.mdtraj.org/)
- [py3Dmol](https://3dmol.org/)
- [OpenMM Cookbook](https://openmm.github.io/openmm-cookbook/)